# SHARPR — Analysis Notebook
Mirrors the original `SHARPR.ipynb` call sequence, updated to import from the split `core` / `viz` / `data` packages instead of the flat `SHARPR_backend` / `SHARPR_viz_alt` / `SHARPR_fetch` files.

Fixes applied vs. the original (see summary for details): removed dead duplicate cells (a stale `IMPROVEMENT_METRICS` dict, a stale single-return `build_improvement_comparisons`, a redundant repeated `evaluate_portfolio_returns_all` call, a `fig_metric_dropdown_barchart()` call missing its required argument), and qualified two private-helper calls that relied on unqualified names.

In [ ]:
#IMPORTS

import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from scipy.optimize import minimize, linprog
from scipy.interpolate import PchipInterpolator
import statsmodels as statsmodels_pkg
import statsmodels.api as sm
import cvxpy as cp

import importlib
from data import fetch
import core as backend
import viz

importlib.reload(fetch)
importlib.reload(backend)
importlib.reload(viz)
# Note: backend/viz are packages built from several submodules via `from .x import *`.
# importlib.reload() only re-executes the package __init__, not its submodules — if you
# edit files under core/ or viz/, restart the kernel to pick up the changes reliably.

## 1. Universe & config

In [ ]:
benchmark_ticker = ['SPY']
risk_free_rate_ticker = ['BIL']

capital = 10_000

#benchmark_ticker = None
#risk_free_rate_ticker = None

tickers = ['OXY','NTR.TO','ASPI','SLV','PYPL','EWY','QQQ','EWJ','INDA','ARGT']

tickers = ['COF', 'AIG', 'PRU', 'GS']
tickers = ['GOOG','AMZN','AAPL','META','MSFT','NFLX','UBER']
tickers = ['GS', 'COST', 'IVN.TO', 'XOM']

tickers = ['FUL','GTLS','KDP','HALO','TBBK','TKO','PCG','CBRE']
tickers = ['COO','HII','PNR','NI','LUV','FE','ON']
# only the last assignment of each block above is actually used — earlier lines are
# left as a quick scratch-pad for swapping in a different universe.

In [ ]:
benchmark_ticker = ['SPY']
_,_, benchmark_ticker, benchmark_ticker_message = fetch.check_tickers(benchmark_ticker)
print(benchmark_ticker_message)

risk_free_rate_ticker = ['BIL']
_,_, risk_free_rate_ticker, risk_free_rate_ticker_message = fetch.check_tickers(risk_free_rate_ticker)
print(risk_free_rate_ticker_message)

_,_, tickers, ticker_message = fetch.check_tickers(tickers)
print(ticker_message)

## 2. Fetch data

In [ ]:
benchmark = fetch.fetch_benchmark(benchmark_ticker)
risk_free_rate = fetch.fetch_risk_free_rate(risk_free_rate_ticker)
prices = fetch.fetch_prices(tickers)
prices = backend.ffill_prices(prices)
factor_df = pd.read_excel("factor_df.xlsx", index_col=0)
factor_df.index = pd.to_datetime(factor_df.index)

## 3. Returns, weights, current/custom portfolios

In [ ]:
benchmark = backend.to_returns(benchmark)
risk_free_rate = backend.to_returns(risk_free_rate)
asset_returns = backend.to_returns(prices)

asset_returns, benchmark, risk_free_rate = backend.cut_complete_asset_sample(asset_returns, benchmark, risk_free_rate)

cov_matrix = backend.get_cov_matrix(asset_returns)
corr_matrix = backend.get_corr_matrix(asset_returns)
sample_means = backend.get_sample_means(asset_returns)
mean_of_sample_means = backend.get_mean_of_sample_means(asset_returns)

current_portfolio_weights = backend.create_random_weights(tickers, seed=42)
custom_portfolio_weights = backend.create_random_weights(tickers, seed=88)

#current_portfolio_weights = None
#custom_portfolio_weights = None

current_portfolio_weights = backend.create_current_portfolio_weights(starting_weights=current_portfolio_weights, returns=asset_returns,) # portfolio_formation_date="2021-11-02")
custom_portfolio_weights = backend.create_current_portfolio_weights(starting_weights=custom_portfolio_weights, returns=asset_returns,) # portfolio_formation_date="2020-07-03")

current_portfolio_return = backend.calculate_drifting_portfolio_return(current_portfolio_weights, asset_returns, "Current Portfolio")
custom_portfolio_return = backend.calculate_drifting_portfolio_return(custom_portfolio_weights, asset_returns, "Custom Portfolio")

## 4. Optimize & aggregate all metrics

In [ ]:
weights_all, info = backend.run_all_optimizers(
    asset_returns=asset_returns,
    rfr=risk_free_rate,      # float daily OR Series/DataFrame daily
    beta=0.95, bound=1.0)

weights_all = backend.add_weights_to_weights_all(
    weights_all,
    current_portfolio_weights=current_portfolio_weights,
    custom_portfolio_weights=custom_portfolio_weights)

portfolio_returns = backend.portfolio_returns_from_weights_all(weights_all, asset_returns)

returns_all = backend.add_reference_returns(
    portfolio_returns,
    current_portfolio_return=current_portfolio_return,
    custom_portfolio_return=custom_portfolio_return,
    benchmark=benchmark, risk_free_rate=risk_free_rate)

rc_pct_all = backend.risk_contribution_pct_all(weights_all, cov_matrix)
sc_pct_all_marg = backend.sharpe_contribution_pct_all(
    weights_all=weights_all,
    er=sample_means,
    cov_matrix=cov_matrix,
    rfr=risk_free_rate,        # can be Series/1-col DF daily, or scalar daily
    rfr_mode="mean",
    rfr_index=asset_returns.index,
    mode="marginal")

sc_pct_all_heur = backend.sharpe_contribution_pct_all(
    weights_all=weights_all,
    er=sample_means,
    cov_matrix=cov_matrix,
    rfr=risk_free_rate,
    rfr_mode="mean",
    rfr_index=asset_returns.index,
    mode="heuristic"
)

loadings_mi, loadings_wide = backend.calc_loadings(returns_all, factor_df, factors=["MKT","HML","SMB","CMA","RMW","MOM"], use_rf=True)

metrics_df = backend.evaluate_portfolio_returns_all(
    portfolio_returns_all=returns_all, asset_returns=asset_returns,
    weights_all=weights_all, cov_matrix=cov_matrix, risk_free_rate=risk_free_rate, beta=0.95)

stats, dd_info = backend.calc_port_stats(
    returns=returns_all,
    benchmark=benchmark,      # 1-col DF
    rfr=risk_free_rate,       # 1-col DF
    required_return_annual=0.0,
    periods=252,
    var_level=5,
    modified_var=True,)

total_df = backend.build_total_performance_metrics_df(metrics_df=metrics_df, stats=stats, prefer="stats")

episodes = backend.drawdown_episodes_panel(returns_all, top_n=5, rank_by="Depth")
episodes_fmt = backend.format_episodes_panel(episodes)

rallies = backend.recovery_rallies_panel(returns_all, top_n=5, rank_by="Recovery Return")
rallies_fmt = backend.format_recovery_rallies_panel(rallies)

bullruns = backend.bullrun_episodes_panel(
    returns_all,
    top_n=5,            # user option
    dd_threshold=0.10,  # “correction” definition (10%)
    rank_by="Bull Return",
)

summary = backend.active_metrics_summary(returns_all=returns_all, benchmark=benchmark, annualization=252)
active_all = backend.active_return_series_all(returns_all=returns_all, benchmark=benchmark, risk_free=risk_free_rate)
effective_bets = backend.effective_n_bets_all(weights_all, cov_matrix, normalize_weights=True)
effective_holdings = backend.effective_n_holdings_all(weights_all, normalize=True)
total_plus_summary = backend.add_to_calc_stats(
    total_df=total_df,
    summary=summary,
    effective_holdings=effective_holdings,
    effective_bets=effective_bets,
    summary_row_prefix=""  # optional (helps avoid duplicate row names)
)

weights_df, dollars_df, delta_df, panel_df = backend.weights_to_dollar_panel(weights_all, capital=capital)
output_dict = backend.make_portfolio_output_dict(panel_df, total_plus_summary, loadings_wide)

wealth_levels = backend.get_wealth_level(returns_all, start_value=1.0)

## 5. Styling

In [ ]:
styles = viz.build_style_registry(
    ticker_names=list(asset_returns.columns),
    portfolio_names=list(returns_all.columns))

## 6. Charts

In [ ]:
fig = viz.plot_return_panel(
    asset_returns=asset_returns,
    benchmark=benchmark,
    risk_free_rate=risk_free_rate,
    custom_portfolio=custom_portfolio_return,
    current_portfolio=current_portfolio_return,
    cumulative=True,
    title="Cumulative Returns of your Portfolio Assets", style_registry=styles)

fig.show()

In [ ]:
fig = viz.plot_cum_returns_and_drawdowns(returns_all, title="SHARPR Portfolios", style_registry=styles)
fig.show()

In [ ]:
fig = viz.plot_efficient_frontier_v2_2(
    asset_returns,
    portfolio_returns_all=returns_all,
    grid="risk",
    annualize=True
)
fig.show()

In [ ]:
fig = viz.plot_allocation_risk_sharpe_switcher(weights_all=weights_all, rc_pct_all=rc_pct_all, sc_pct_all_heur=sc_pct_all_heur, style_registry=styles)
fig.show()

In [ ]:
fig = viz.fig_capital_reallocation_bars(
    weights_all,
    capital=capital,
    current_portfolio_name="Current Portfolio",
    show="delta_if_possible",  # default behavior you wanted
    styles=styles,
    as_pct=False                # show dollars
)
fig.show()

In [ ]:
fig = viz.plot_loadings_wide(
    loadings_wide,
    annualize_alpha=True,
    alpha_unit="decimal",
    legend_x=1,
    right_margin=0.6, style_registry=styles
)
fig.show()

In [ ]:
fig, out = viz.simulate_and_plot_two_gbm(
    portfolio_returns_all=returns_all,
    sim_cols=["Max. Sharpe Ratio", "Max. Diversification"],  # pick up to two columns here
    n_paths=1000,
    horizon_days=252,
    plot_paths=250,
    use_last_for_sim=756,   # last ~3 years of daily returns (set None for full sample)
    plot_last_t=800,        # only show last 800 dates in the top panel (wealth still cumprod from start)
    seed=1, style_registry=styles, legend_x=1.02,
    right_margin=90)

fig.show()

In [ ]:
fig = viz.fig_wealth_drawdown_panel_viewer_tabbed(
    returns_all=returns_all,
    episodes_df=episodes_fmt,
    rallies_df=rallies_fmt,
    bullruns_df=bullruns,
    styles=styles,
    portfolio="Max. Sharpe Ratio",
    panel_kind="rallies",
    include_risk_free=False,
    width=1200,
    height=1180,
)

fig.show()

In [ ]:
fig = viz.plot_scorecard_grid_v2(total_plus_summary,
    portfolios=None, initial_preset="balanced", title="Portfolio Scorecard",)
fig.show()

In [ ]:
# Gain vs pain: x/y required; size/color optional
fig2 = viz.plot_gain_pain_scatter(
    total_plus_summary,
    x_metric="Annualized Mean Return",     # will default-transform to abs() so “depth” is positive
    y_metric="Annualized Volatility",
    z_metric='Expected Shortfall',
    size_metric="Diversification Ratio",
    color_metric="Sharpe Ratio",
    title="Gain vs Pain Chart",
    styles=styles
)
fig2.show()

In [ ]:
fig_fwd = viz.fig_forward_horizon_distribution(returns_all, horizons=(5, 21),)
fig_fwd.show()

In [ ]:
# Raw levels
fig = viz.fig_metric_barchart(total_plus_summary.T, metric="Sharpe Ratio", baseline=None, mode="level", styles=styles)  # metric and mode changeable
fig.show()

# % difference vs baseline portfolio
#fig = viz.fig_metric_barchart(total_df, metric="Sharpe Ratio", baseline="Risk Parity", mode="pct", styles=styles)
#fig.show()

# Absolute difference vs baseline
#fig = viz.fig_metric_barchart(total_df, metric="Total Risk", baseline="Min. Variance", mode="abs", styles=styles)
#fig.show()

In [ ]:
fig2 = viz.fig_metric_dropdown_barchart(total_plus_summary.T, metrics=total_df.columns, mode='level', styles=styles)
fig2.show()

## 7. Export to Excel

In [ ]:
filepath = backend.export_output_dict_to_excel(
    output_dict=output_dict,
    tickers=tickers,
    for_streamlit=False
)

## 8. Drawdown summary & portfolio improvement comparison

In [ ]:
def drawdown_summary_metrics(returns):
    r = backend._as_df(returns)
    dd, _, _ = backend.drawdown_series(r)

    out = {}

    for col in dd.columns:
        spells = backend._drawdown_spells_one(dd[col])

        if spells.empty:
            out[col] = {
                "Average Drawdown": np.nan,
                "Average Underwater Period": np.nan,
                "Longest Underwater Period": np.nan,
                "Average Recovery Period": np.nan,
                "Longest Recovery Period": np.nan,
            }
            continue

        recovered = spells.loc[spells["Recovered"]].copy()

        out[col] = {
            "Average Drawdown": -pd.to_numeric(spells["Depth"], errors="coerce").mean(),
            "Average Underwater Period": pd.to_numeric(spells["Underwater Days"], errors="coerce").mean(),
            "Longest Underwater Period": pd.to_numeric(spells["Underwater Days"], errors="coerce").max(),
            "Average Recovery Period": pd.to_numeric(recovered["Valley→Recovery Days"], errors="coerce").mean(),
            "Longest Recovery Period": pd.to_numeric(recovered["Valley→Recovery Days"], errors="coerce").max(),
        }

    return pd.DataFrame(out)

# note: the original notebook called _as_df / drawdown_series / _drawdown_spells_one
# unqualified here, which would NameError on a fresh kernel since only `backend` itself
# is imported (not its private members) — qualified with `backend.` above.

In [ ]:
drawdown_summary = drawdown_summary_metrics(returns_all)
total_plus_summary = pd.concat([total_plus_summary, drawdown_summary,], axis=0,)

In [ ]:
IMPROVEMENT_METRICS = {

    "Max. Diversification": {
        "primary": [
            "Diversification Ratio",
        ],
        "secondary": [
            "Effective Holdings",
            "Effective Bets",
            "Annualized Volatility",
            "Max. Drawdown",
            "Expected Shortfall",
        ],
        "cost": [
            "Annualized Mean Return",
            "Sharpe Ratio",
            "Portfolio Turnover",
            "Tracking Error",
        ],
    },

    "Max. Sharpe Ratio": {
        "primary": [
            "Sharpe Ratio",
        ],
        "secondary": [
            "Annualized Mean Return",
            "Annualized Volatility",
            "Sortino Ratio",
            "Max. Drawdown",
            "Expected Shortfall",
        ],
        "cost": [
            "Effective Holdings",
            "Effective Bets",
            "Portfolio Turnover",
            "Tracking Error",
        ],
    },

    "Min. Variance": {
        "primary": [
            "Annualized Volatility",
        ],
        "secondary": [
            "Semi-Deviation",
            "Expected Shortfall",
            "Max. Drawdown",
            "Ulcer Index",
            "Risk of Ruin (q/p)^15",
        ],
        "cost": [
            "Annualized Mean Return",
            "Sharpe Ratio",
            "Effective Holdings",
            "Portfolio Turnover",
        ],
    },

    "Risk Parity": {
        "primary": [
            "Risk Contribution Dispersion",
        ],
        "secondary": [
            "Diversification Ratio",
            "Effective Holdings",
            "Effective Bets",
            "Annualized Volatility",
            "Max. Drawdown",
        ],
        "cost": [
            "Annualized Mean Return",
            "Sharpe Ratio",
            "Portfolio Turnover",
            "Tracking Error",
        ],
    },

    "Min. Expected Shortfall": {
        "primary": [
            "Expected Shortfall",
        ],
        "secondary": [
            "Conditional VaR (5%)",
            "Historical VaR (5%)",
            "Worst Day",
            "Semi-Deviation",
            "Max. Drawdown",
        ],
        "cost": [
            "Annualized Mean Return",
            "Sharpe Ratio",
            "Annualized Volatility",
            "Portfolio Turnover",
        ],
    },

    "Min. Drawdown": {
        "primary": [
            "Max. Drawdown",
        ],
        "secondary": [
            "Ulcer Index",
            "Average Drawdown",
            "Longest Underwater Period",
            "Average Underwater Days",
            "Expected Shortfall",
        ],
        "cost": [
            "Annualized Mean Return",
            "Sharpe Ratio",
            "Annualized Volatility",
            "Portfolio Turnover",
        ],
    },
}
# note: the original notebook defined this dict twice (an earlier, slightly different
# version got silently overwritten); only the final/live version is kept here.

In [ ]:
def build_improvement_comparisons(total_plus_summary, improvement_metrics):
    out = {}
    out_extended = {}

    benchmark_cols = [
        c for c in total_plus_summary.columns
        if "benchmark" in str(c).lower()
    ]

    benchmark_col = benchmark_cols[0] if len(benchmark_cols) > 0 else None

    current_col = (
        "Current Portfolio"
        if "Current Portfolio" in total_plus_summary.columns
        else None
    )

    custom_col = (
        "Custom Portfolio"
        if "Custom Portfolio" in total_plus_summary.columns
        else None
    )

    for portfolio, config in improvement_metrics.items():

        if portfolio not in total_plus_summary.columns:
            continue

        improvement = config["primary"] + config["secondary"]
        tradeoff = config["cost"]
        metrics = improvement + tradeoff
        metrics = [m for m in metrics if m in total_plus_summary.index]

        cols = []
        if current_col is not None:
            cols.append(current_col)
        if custom_col is not None:
            cols.append(custom_col)
        if benchmark_col is not None:
            cols.append(benchmark_col)
        cols.append(portfolio)
        cols = list(dict.fromkeys(cols))

        df = total_plus_summary.loc[metrics, cols].copy()

        df_extended = df.copy()
        df_extended["Metric Type"] = [
            "Improvement" if metric in improvement else "Tradeoff"
            for metric in df_extended.index
        ]

        out[portfolio] = df
        out_extended[portfolio] = df_extended

    return out, out_extended

# note: the original notebook defined this function twice — an earlier version
# returned only `out` (a single dict); the call site further down was written against
# that earlier signature and, unpacked against this final two-return version, would
# have produced a wrong/stale result. Only the final two-return version is kept,
# together with the matching two-value call below.

In [ ]:
improvement_comparisons, improvement_comparisons_extended = build_improvement_comparisons(
    total_plus_summary,
    IMPROVEMENT_METRICS
)

In [ ]:
improvement_comparisons_extended['Max. Diversification']

In [ ]:
improvement_comparisons.keys()

In [ ]:
improvement_comparisons["Max. Sharpe Ratio"]

## 9. Custom charts (notebook-only — not part of the `viz` package)

In [ ]:
def plot_improvement_comparison(improvement_comparisons_extended):

    strategies = list(improvement_comparisons_extended.keys())

    if len(strategies) == 0:
        raise ValueError("improvement_comparisons_extended is empty.")

    strategy_info = {}
    max_improvement = 0
    max_tradeoff = 0

    for strategy in strategies:
        df = improvement_comparisons_extended[strategy].copy()
        metric_type = df["Metric Type"]
        values = df.drop(columns="Metric Type")
        primary_metric = df.index[0]

        improvement_metrics = [m for m in df.index[1:] if metric_type.loc[m] == "Improvement"]
        tradeoff_metrics = [m for m in df.index[1:] if metric_type.loc[m] == "Tradeoff"]

        strategy_info[strategy] = {
            "df": df,
            "values": values,
            "primary": primary_metric,
            "improvement": improvement_metrics,
            "tradeoff": tradeoff_metrics,
        }

        max_improvement = max(max_improvement, len(improvement_metrics))
        max_tradeoff = max(max_tradeoff, len(tradeoff_metrics))

    n_metric_rows = max(max_improvement, max_tradeoff)
    n_rows = 2 + n_metric_rows

    row_heights = [0.24] + [0.08] + [0.68 / n_metric_rows] * n_metric_rows
    specs = [[{"colspan": 2}, None]] + [[None, None]] + [[{}, {}] for _ in range(n_metric_rows)]

    fig = make_subplots(
        rows=n_rows, cols=2, specs=specs, row_heights=row_heights,
        vertical_spacing=0.065, horizontal_spacing=0.15,
    )

    strategy_trace_indices = {}

    for strategy_i, strategy in enumerate(strategies):
        info = strategy_info[strategy]
        values = info["values"]
        primary_metric = info["primary"]
        improvement_metrics = info["improvement"]
        tradeoff_metrics = info["tradeoff"]
        visible = strategy_i == 0
        strategy_trace_indices[strategy] = []

        primary_values = pd.to_numeric(values.loc[primary_metric], errors="coerce").dropna()
        fig.add_trace(
            go.Bar(
                x=primary_values.values, y=primary_values.index, orientation="h",
                text=[f"{x:.3f}" for x in primary_values.values],
                textposition="outside", showlegend=False, visible=visible,
                hovertemplate="%{y}: %{x:.4f}<extra></extra>",
            ),
            row=1, col=1,
        )
        strategy_trace_indices[strategy].append(len(fig.data) - 1)

        for i, metric in enumerate(improvement_metrics):
            metric_values = pd.to_numeric(values.loc[metric], errors="coerce").dropna()
            fig.add_trace(
                go.Scatter(
                    x=metric_values.values, y=metric_values.index, mode="markers+text",
                    text=[f"{x:.3f}" for x in metric_values.values],
                    textposition="middle right", marker=dict(size=10),
                    showlegend=False, visible=visible,
                    hovertemplate="%{y}: %{x:.4f}<extra></extra>",
                ),
                row=i + 3, col=1,
            )
            strategy_trace_indices[strategy].append(len(fig.data) - 1)

        for i, metric in enumerate(tradeoff_metrics):
            metric_values = pd.to_numeric(values.loc[metric], errors="coerce").dropna()
            fig.add_trace(
                go.Scatter(
                    x=metric_values.values, y=metric_values.index, mode="markers+text",
                    text=[f"{x:.3f}" for x in metric_values.values],
                    textposition="middle right", marker=dict(size=10),
                    showlegend=False, visible=visible,
                    hovertemplate="%{y}: %{x:.4f}<extra></extra>",
                ),
                row=i + 3, col=2,
            )
            strategy_trace_indices[strategy].append(len(fig.data) - 1)

    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(showgrid=True, zeroline=False)

    fig.update_layout(
        height=420 + 165 * n_metric_rows,
        margin=dict(l=110, r=80, t=150, b=50),
        title=dict(text="Portfolio Improvement Comparison", x=0.5),
        hovermode="closest",
    )

    def make_annotations(strategy):
        info = strategy_info[strategy]
        primary_metric = info["primary"]
        improvement_metrics = info["improvement"]
        tradeoff_metrics = info["tradeoff"]
        annotations = []

        primary_subplot = fig.get_subplot(1, 1)
        annotations.append(dict(
            text=f"<b>{primary_metric}</b>", x=0.5,
            y=primary_subplot.yaxis.domain[1] + 0.015,
            xref="paper", yref="paper", xanchor="center", yanchor="bottom",
            showarrow=False, font=dict(size=17),
        ))

        first_left = fig.get_subplot(3, 1)
        first_right = fig.get_subplot(3, 2)
        section_y = max(first_left.yaxis.domain[1], first_right.yaxis.domain[1]) + 0.045

        annotations.append(dict(
            text="<b>IMPROVEMENT METRICS</b>", x=first_left.xaxis.domain[0], y=section_y,
            xref="paper", yref="paper", xanchor="left", yanchor="bottom",
            showarrow=False, font=dict(size=13),
        ))
        annotations.append(dict(
            text="<b>TRADE-OFF METRICS</b>", x=first_right.xaxis.domain[0], y=section_y,
            xref="paper", yref="paper", xanchor="left", yanchor="bottom",
            showarrow=False, font=dict(size=13),
        ))

        for i, metric in enumerate(improvement_metrics):
            subplot = fig.get_subplot(i + 3, 1)
            annotations.append(dict(
                text=f"<b>{metric}</b>", x=subplot.xaxis.domain[0],
                y=subplot.yaxis.domain[1] + 0.012,
                xref="paper", yref="paper", xanchor="left", yanchor="bottom",
                showarrow=False, font=dict(size=14),
            ))

        for i, metric in enumerate(tradeoff_metrics):
            subplot = fig.get_subplot(i + 3, 2)
            annotations.append(dict(
                text=f"<b>{metric}</b>", x=subplot.xaxis.domain[0],
                y=subplot.yaxis.domain[1] + 0.012,
                xref="paper", yref="paper", xanchor="left", yanchor="bottom",
                showarrow=False, font=dict(size=14),
            ))

        return annotations

    initial_strategy = strategies[0]
    fig.update_layout(annotations=make_annotations(initial_strategy))

    buttons = []
    for strategy in strategies:
        visible = [False] * len(fig.data)
        for trace_i in strategy_trace_indices[strategy]:
            visible[trace_i] = True
        buttons.append(dict(
            label=strategy, method="update",
            args=[{"visible": visible}, {"annotations": make_annotations(strategy)}],
        ))

    fig.update_layout(
        updatemenus=[dict(
            type="dropdown", direction="down", x=0.0, y=1.11,
            xanchor="left", yanchor="top", buttons=buttons, showactive=True,
        )]
    )

    fig.add_annotation(
        text="<b>Optimization Strategy:</b>", x=0.0, y=1.135,
        xref="paper", yref="paper", xanchor="left", yanchor="bottom",
        showarrow=False, font=dict(size=13),
    )

    return fig

In [ ]:
fig = plot_improvement_comparison(improvement_comparisons_extended)
fig.show()

In [ ]:
def plot_return_correlation_heatmap(asset_returns, benchmark=None, risk_free_rate=None):

    returns = asset_returns.copy()
    if isinstance(returns, pd.Series):
        returns = returns.to_frame()

    if benchmark is not None:
        benchmark = benchmark.copy()
        if isinstance(benchmark, pd.DataFrame):
            benchmark = benchmark.iloc[:, 0]
        benchmark.name = "Benchmark"
        returns = pd.concat([returns, benchmark], axis=1)

    if risk_free_rate is not None:
        risk_free_rate = risk_free_rate.copy()
        if isinstance(risk_free_rate, pd.DataFrame):
            risk_free_rate = risk_free_rate.iloc[:, 0]
        risk_free_rate.name = "Risk Free Rate"
        returns = pd.concat([returns, risk_free_rate], axis=1)

    returns = returns.dropna()
    corr = returns.corr()

    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    corr_lower = corr.mask(mask)

    text = corr_lower.copy()
    for col in text.columns:
        text[col] = text[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else "")

    fig = go.Figure(
        data=go.Heatmap(
            z=corr_lower.values, x=corr_lower.columns, y=corr_lower.index,
            zmin=-1, zmax=1, zmid=0,
            colorscale=[[0.0, "#b2182b"], [0.5, "#ffffff"], [1.0, "#1a9850"]],
            text=text.values, texttemplate="%{text}",
            hovertemplate="%{y} / %{x}<br>Correlation: %{z:.2f}<extra></extra>",
            colorbar=dict(title="Correlation", tickvals=[-1, -0.5, 0, 0.5, 1]),
            hoverongaps=False,
        )
    )

    n = len(corr.columns)
    fig.update_layout(
        title=dict(text="How Your Investments Move Together", x=0.5),
        xaxis=dict(side="bottom", tickangle=-45, showgrid=False),
        yaxis=dict(autorange="reversed", showgrid=False),
        height=max(500, 48 * n + 150),
        margin=dict(l=110, r=80, t=80, b=110),
    )

    return fig

In [ ]:
fig = plot_return_correlation_heatmap(asset_returns, None, None)
fig.show()